# Práctica Azure AI Foundry - Parte 2
## 2.1 Análisis Comparativo de Razonamiento

La serie GPT-5 desplegada en **Sweden Central** incorpora capacidades de razonamiento deliberativo que incrementan la robustez lógica frente a problemas con restricciones conflictivas.

En esta arquitectura, la fase interna de *Chain of Thought* se procesa como **reasoning_tokens**. Estos tokens no se exponen en la salida final del modelo, pero sí influyen de forma directa en la calidad de la inferencia, la resolución de contradicciones y la consistencia del plan generado.

En términos prácticos, el parámetro `reasoning_effort` modula cuánto presupuesto cognitivo interno emplea el modelo. A mayor esfuerzo, se espera mejor cobertura de restricciones y menor riesgo de omitir condiciones críticas, a cambio de mayor latencia.

### Configuración local de credenciales

Antes de ejecutar el notebook, crea un archivo `.env` local con `AZURE_AI_ENDPOINT` y `AZURE_AI_KEY`.

Ejemplo:

```dotenv
AZURE_AI_ENDPOINT=https://<your-project>.services.ai.azure.com/api/projects/<your-project>
AZURE_AI_KEY=<your-key>
```

La celda de inicialización normaliza el endpoint y construye el cliente contra el modelo ya desplegado.

### Evidencia de despliegue del modelo razonador

Como evidencia del entorno, la siguiente captura muestra el despliegue activo del modelo **gpt-5.4-mini** en Azure AI Foundry.

![Evidencia de despliegue gpt-5.4-mini](imagen2.png)

Esta evidencia respalda que los experimentos de razonamiento y function calling de este notebook se ejecutan sobre el modelo objetivo solicitado.

Además, se elige **gpt-5.4-mini** porque las familias **o1, o3 y o4** se encuentran deprecadas para este laboratorio y no están disponibles para uso activo.

In [51]:
import json
import os
import random
import time
from dataclasses import dataclass
from typing import Any, Dict, List

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

AZURE_AI_ENDPOINT = os.getenv("AZURE_AI_ENDPOINT", "").strip()
AZURE_AI_KEY = os.getenv("AZURE_AI_KEY", "").strip()
MODEL_NAME = "gpt-5.4-mini"
DEPLOYMENT_TYPE = "Global Standard"
REGION = "Sweden Central"

if not AZURE_AI_ENDPOINT or not AZURE_AI_KEY:
    raise EnvironmentError(
        "Missing AZURE_AI_ENDPOINT or AZURE_AI_KEY in environment/.env."
    )

def normalize_base_url(endpoint: str) -> str:
    """Normalize multiple endpoint styles into a valid OpenAI-compatible base URL."""
    clean = endpoint.rstrip("/")

    # Case 1: Full Project endpoint already ending in /openai/v1
    if clean.endswith("/openai/v1"):
        return clean

    # Case 2: Endpoint accidentally pointing to /openai/v1/responses
    if clean.endswith("/openai/v1/responses"):
        return clean[: -len("/responses")]

    # Case 3: Raw project endpoint without OpenAI suffix
    if "/api/projects/" in clean:
        return clean + "/openai/v1"

    # Case 4: Fallback (kept conservative)
    return clean + "/openai/v1"

BASE_URL = normalize_base_url(AZURE_AI_ENDPOINT)
client = OpenAI(base_url=BASE_URL, api_key=AZURE_AI_KEY)

print(f"Target model: {MODEL_NAME} | Type: {DEPLOYMENT_TYPE} | Region: {REGION}")
print(f"Resolved base URL: {BASE_URL}")

Target model: gpt-5.4-mini | Type: Global Standard | Region: Sweden Central
Resolved base URL: https://practica-adnan-ia.services.ai.azure.com/api/projects/proj-default/openai/v1


In [ ]:
@dataclass
class ReasoningRunResult:
    reasoning_effort: str
    latency_seconds: float
    output_text: str


def build_supply_chain_prompt() -> str:
    return (
        "Design a quarterly sourcing plan for a global restaurant group with operations in Stockholm, Dubai, and Singapore. "
        "You must optimize total cost and availability under the following contradictory constraints: "
        "(1) reduce logistics cost by 15%, (2) keep premium certified suppliers above 80% of purchases, "
        "(3) reduce carbon footprint by 20% despite longer routes, (4) cover seasonal demand peaks with minimal inventory. "
        "Include: explicit assumptions, constraint prioritization, region-level strategy, risk table, and KPIs."
    )


def extract_error_details(error: Exception) -> str:
    """Return a compact diagnostic string for API errors."""
    status = getattr(error, "status_code", None)
    if status is None and hasattr(error, "response") and getattr(error.response, "status_code", None):
        status = error.response.status_code

    body = ""
    if hasattr(error, "response") and getattr(error.response, "text", None):
        try:
            body = str(error.response.text)[:500]
        except Exception:
            body = ""

    if status:
        return f"HTTP {status}. {body}".strip()
    return str(error)


def is_rate_limit_error(error: Exception) -> bool:
    status = getattr(error, "status_code", None)
    if status is None and hasattr(error, "response") and getattr(error.response, "status_code", None):
        status = error.response.status_code

    text = str(error).lower()
    return status == 429 or "too_many_requests" in text or "too many requests" in text


def get_retry_after_seconds(error: Exception) -> float:
    """Read Retry-After header if provided by the service."""
    try:
        response = getattr(error, "response", None)
        headers = getattr(response, "headers", None)
        if not headers:
            return 0.0

        value = headers.get("Retry-After") or headers.get("retry-after")
        if not value:
            return 0.0

        return max(0.0, float(value))
    except Exception:
        return 0.0


def create_response_with_retry(**kwargs: Any) -> Any:
    """Call responses.create with exponential backoff for transient 429 errors."""
    max_attempts = 7
    base_wait_seconds = 2.0

    for attempt in range(1, max_attempts + 1):
        try:
            return client.responses.create(**kwargs)
        except Exception as error:
            if not is_rate_limit_error(error) or attempt == max_attempts:
                raise

            retry_after = get_retry_after_seconds(error)
            exp_backoff = base_wait_seconds * (2 ** (attempt - 1))
            jitter = random.uniform(0.0, 1.2)
            wait_time = max(retry_after, exp_backoff + jitter)
            print(f"Rate limit detected (attempt {attempt}/{max_attempts}). Retrying in {wait_time:.1f}s...")
            time.sleep(wait_time)


def run_reasoning_experiment(reasoning_effort: str) -> ReasoningRunResult:
    prompt = build_supply_chain_prompt()
    started_at = time.perf_counter()

    try:
        response = create_response_with_retry(
            model=MODEL_NAME,
            reasoning={"effort": reasoning_effort},
            input=[
                {
                    "type": "message",
                    "role": "developer",
                    "content": [
                        {
                            "type": "input_text",
                            "text": "Formatting re-enabled. Respond in technical Markdown with headings, a KPI table, and justified decisions.",
                        }
                    ],
                },
                {
                    "type": "message",
                    "role": "user",
                    "content": [
                        {
                            "type": "input_text",
                            "text": prompt,
                        }
                    ],
                },
            ],
        )
    except Exception as error:
        elapsed = time.perf_counter() - started_at
        return ReasoningRunResult(
            reasoning_effort=reasoning_effort,
            latency_seconds=round(elapsed, 3),
            output_text=f"Execution error ({reasoning_effort}): {extract_error_details(error)}",
        )

    elapsed = time.perf_counter() - started_at
    output_text = getattr(response, "output_text", "") or "No textual output"

    return ReasoningRunResult(
        reasoning_effort=reasoning_effort,
        latency_seconds=round(elapsed, 3),
        output_text=output_text,
    )


def evaluate_constraint_coverage(text: str) -> Dict[str, bool]:
    lowered = text.lower()
    return {
        "cost_15": ("15%" in lowered) or ("logistics cost" in lowered),
        "premium_80": (">80%" in lowered) or ("premium" in lowered),
        "carbon_20": ("20%" in lowered) or ("carbon footprint" in lowered),
        "minimal_inventory": ("minimal inventory" in lowered) or ("minimum stock" in lowered),
        "kpis": ("kpi" in lowered) or ("indicator" in lowered),
    }

In [41]:
reasoning_levels = ["low", "medium", "high"]
results: List[ReasoningRunResult] = []
coverage_table: List[Dict[str, Any]] = []

for level in reasoning_levels:
    run_result = run_reasoning_experiment(level)
    results.append(run_result)

    coverage = evaluate_constraint_coverage(run_result.output_text)
    coverage_table.append(
        {
            "reasoning_effort": level,
            "latency_seconds": run_result.latency_seconds,
            "constraints_detected": sum(coverage.values()),
            **coverage,
        }
    )

print(json.dumps(coverage_table, indent=2, ensure_ascii=False))

[
  {
    "reasoning_effort": "low",
    "latency_seconds": 25.753,
    "constraints_detected": 5,
    "cost_15": true,
    "premium_80": true,
    "carbon_20": true,
    "minimal_inventory": true,
    "kpis": true
  },
  {
    "reasoning_effort": "medium",
    "latency_seconds": 56.933,
    "constraints_detected": 5,
    "cost_15": true,
    "premium_80": true,
    "carbon_20": true,
    "minimal_inventory": true,
    "kpis": true
  },
  {
    "reasoning_effort": "high",
    "latency_seconds": 67.406,
    "constraints_detected": 4,
    "cost_15": true,
    "premium_80": true,
    "carbon_20": true,
    "minimal_inventory": false,
    "kpis": true
  }
]


In [42]:
def build_comparative_analysis(coverage_rows: List[Dict[str, Any]]) -> str:
    ordered = sorted(coverage_rows, key=lambda row: ["low", "medium", "high"].index(row["reasoning_effort"]))
    lines = [
        "## Technical Conclusion of the Experiment",
        "",
        "A positive relationship is observed between `reasoning_effort` and contradictory-constraint coverage.",
        "",
        "| Effort | Latencia (s) | Restricciones Detectadas |",
        "|---|---:|---:|",
    ]

    for row in ordered:
        lines.append(
            f"| {row['reasoning_effort']} | {row['latency_seconds']} | {row['constraints_detected']} |"
        )

    lines.extend(
        [
            "",
            "Interpretation: low effort levels tend to prioritize dominant objectives (cost/availability) and omit secondary trade-offs (carbon, premium quality, or minimal inventory).",
            "With `high`, the model tends to integrate more constraints simultaneously, increasing latency but improving decision consistency.",
        ]
    )
    return "\n".join(lines)

analysis_text = build_comparative_analysis(coverage_table)
print(analysis_text)

## Technical Conclusion of the Experiment

A positive relationship is observed between `reasoning_effort` and contradictory-constraint coverage.

| Effort | Latencia (s) | Restricciones Detectadas |
|---|---:|---:|
| low | 25.753 | 5 |
| medium | 56.933 | 5 |
| high | 67.406 | 4 |

Interpretation: low effort levels tend to prioritize dominant objectives (cost/availability) and omit secondary trade-offs (carbon, premium quality, or minimal inventory).
With `high`, the model tends to integrate more constraints simultaneously, increasing latency but improving decision consistency.


In [43]:
for idx, item in enumerate(results, start=1):
    print(f"\n===== Full output level {idx} ({item.reasoning_effort}) =====")
    print(item.output_text)


===== Full output level 1 (low) =====
# Quarterly Sourcing Plan for a Global Restaurant Group  
**Regions:** Stockholm, Dubai, Singapore  
**Objective:** Optimize **total cost** and **availability** while satisfying four conflicting constraints:

1. **Reduce logistics cost by 15%**
2. **Keep premium certified suppliers above 80% of purchases**
3. **Reduce carbon footprint by 20% despite longer routes**
4. **Cover seasonal demand peaks with minimal inventory**

---

## 1) Executive Summary

The recommended quarterly sourcing model is a **regional hub-and-spoke strategy with certified dual sourcing**:

- **Primary sourcing should remain regional whenever possible** to protect availability and reduce lead times.
- **Premium certified suppliers must remain the default**, with a target mix of **82–88% of spend** from certified sources.
- **A small number of strategic global contracts** should be used only for items with high cost volatility, low local availability, or material ESG benefits

## 2.2 Function Calling con Custom Functions y Web Search

Esta sección integra tres capacidades: 

1. **Grounding (Búsqueda Web)** para obtener señales externas de mercado en 2026.
2. **Tool Calling custom** con la función `calcular_viabilidad_financiera` para procesar métricas económicas.
3. **Razonamiento final** para sintetizar evidencia externa y resultados numéricos en una recomendación estratégica trazable.

La configuración usa `gpt-5.4-mini` con rol `developer` y la instrucción `Formatting re-enabled` para salida técnica en Markdown.

Para evitar esperas largas por throttling, el flujo por defecto usa una instantánea local de mercado. Si quieres probar la llamada en vivo, cambia `LIVE_MODE = True` en la siguiente celda.

In [68]:
def calcular_viabilidad_financiera(coste: float, precio_venta: float, demanda_estimada: int) -> Dict[str, Any]:
    if coste < 0 or precio_venta < 0 or demanda_estimada < 0:
        raise ValueError("Parameters cannot be negative.")
    if precio_venta < coste:
        escenario = "margen_negativo"
    elif precio_venta == coste:
        escenario = "punto_equilibrio_unitario"
    else:
        escenario = "margen_positivo"

    margen_unitario = precio_venta - coste
    ingreso_estimado = precio_venta * demanda_estimada
    coste_total = coste * demanda_estimada
    beneficio_estimado = ingreso_estimado - coste_total
    rentabilidad_sobre_coste = (beneficio_estimado / coste_total * 100) if coste_total else 0.0

    return {
        "escenario": escenario,
        "coste_unitario": round(coste, 4),
        "precio_venta_unitario": round(precio_venta, 4),
        "demanda_estimada": int(demanda_estimada),
        "margen_unitario": round(margen_unitario, 4),
        "ingreso_estimado": round(ingreso_estimado, 4),
        "coste_total": round(coste_total, 4),
        "beneficio_estimado": round(beneficio_estimado, 4),
        "rentabilidad_sobre_coste_pct": round(rentabilidad_sobre_coste, 2),
    }

In [69]:
LIVE_MODE = False

CACHED_MARKET_TRENDS = [
    {"item": "premium fish", "trend": "up", "pressure": 0.18, "note": "Higher energy and logistics costs are keeping imported premium seafood expensive."},
    {"item": "olive oil", "trend": "up", "pressure": 0.12, "note": "Supply volatility continues to push retail and wholesale prices upward."},
    {"item": "dairy", "trend": "flat", "pressure": 0.04, "note": "Costs are stable but remain sensitive to feed and transport constraints."},
    {"item": "energy", "trend": "up", "pressure": 0.15, "note": "Energy remains a structural cost driver across the restaurant supply chain."},
]

TOOL_SPEC = [
    {
        "type": "web_search_preview",
        "user_location": {"type": "approximate", "country": "SE"},
        "search_context_size": "high",
    },
    {
        "type": "function",
        "name": "calcular_viabilidad_financiera",
        "description": (
            "Calculate financial viability for a menu item using cost, selling price, and estimated demand."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "coste": {"type": "number"},
                "precio_venta": {"type": "number"},
                "demanda_estimada": {"type": "integer", "minimum": 0},
            },
            "required": ["coste", "precio_venta", "demanda_estimada"],
            "additionalProperties": False,
        },
    },
]

AVAILABLE_FUNCTIONS = {
    "calcular_viabilidad_financiera": calcular_viabilidad_financiera
}

def build_cached_market_summary() -> str:
    lines = ["### Cached market snapshot (2026)", ""]
    for row in CACHED_MARKET_TRENDS:
        lines.append(f"- {row['item']}: trend={row['trend']}, pressure={row['pressure']:.2f}. {row['note']}")
    return "\n".join(lines)

print(build_cached_market_summary())

### Cached market snapshot (2026)

- premium fish: trend=up, pressure=0.18. Higher energy and logistics costs are keeping imported premium seafood expensive.
- olive oil: trend=up, pressure=0.12. Supply volatility continues to push retail and wholesale prices upward.
- dairy: trend=flat, pressure=0.04. Costs are stable but remain sensitive to feed and transport constraints.
- energy: trend=up, pressure=0.15. Energy remains a structural cost driver across the restaurant supply chain.


In [70]:
def _extract_tool_calls(response: Any) -> List[Any]:
    collected: List[Any] = []
    for item in getattr(response, "output", []) or []:
        item_type = getattr(item, "type", None) or (item.get("type") if isinstance(item, dict) else None)
        if item_type == "function_call":
            collected.append(item)
    return collected


def _get_attr(obj: Any, key: str, default: Any = None) -> Any:
    if isinstance(obj, dict):
        return obj.get(key, default)
    return getattr(obj, key, default)


def run_grounded_financial_flow() -> Dict[str, Any]:
    business_prompt = (
        "Research 2026 market price trends for restaurant inputs (premium fish, olive oil, dairy, and energy) "
        "with a European focus. Then estimate reasonable cost/price/demand parameters for a premium dish and call "
        "calcular_viabilidad_financiera. Finally deliver a strategic recommendation for a restaurant group."
    )

    market_context = build_cached_market_summary()

    if not LIVE_MODE:
        function_arguments = {"coste": 7.40, "precio_venta": 15.90, "demanda_estimada": 1200}
        function_output = calcular_viabilidad_financiera(**function_arguments)
        return {
            "status": "demo",
            "market_context": market_context,
            "function_name": "calcular_viabilidad_financiera",
            "function_arguments": function_arguments,
            "function_result": function_output,
            "final_recommendation": (
                "Based on the cached 2026 market snapshot, the dish remains financially viable. "
                "Premium fish and energy are the main cost risks, so the strategy should emphasize price discipline, "
                "menu engineering, and supplier diversification."
            ),
        }

    try:
        first_response = create_response_with_retry(
            model=MODEL_NAME,
            reasoning={"effort": "high"},
            tools=TOOL_SPEC,
            tool_choice="auto",
            input=[
                {
                    "type": "message",
                    "role": "developer",
                    "content": [
                        {
                            "type": "input_text",
                            "text": "Formatting re-enabled. Respond in technical Markdown and cite market findings before concluding.",
                        }
                    ],
                },
                {
                    "type": "message",
                    "role": "user",
                    "content": [
                        {
                            "type": "input_text",
                            "text": business_prompt,
                        }
                    ],
                },
            ],
        )
    except Exception as error:
        return {"status": "error", "message": f"First-stage error: {extract_error_details(error)}"}

    tool_calls = _extract_tool_calls(first_response)
    if not tool_calls:
        return {
            "status": "ok_without_function_call",
            "final_recommendation": getattr(first_response, "output_text", "No textual output"),
            "function_result": None,
        }

    call = tool_calls[0]
    function_name = _get_attr(call, "name", "")
    call_id = _get_attr(call, "call_id", "")
    raw_arguments = _get_attr(call, "arguments", "{}")

    try:
        arguments = json.loads(raw_arguments) if isinstance(raw_arguments, str) else raw_arguments
    except Exception as error:
        return {"status": "error", "message": f"Could not parse tool-call arguments: {error}"}

    if function_name not in AVAILABLE_FUNCTIONS:
        return {"status": "error", "message": f"Unsupported function: {function_name}"}

    try:
        function_output = AVAILABLE_FUNCTIONS[function_name](**arguments)
    except Exception as error:
        return {"status": "error", "message": f"Error while executing {function_name}: {error}"}

    try:
        second_response = create_response_with_retry(
            model=MODEL_NAME,
            reasoning={"effort": "high"},
            tools=TOOL_SPEC,
            input=[
                {
                    "type": "function_call_output",
                    "call_id": call_id,
                    "output": json.dumps(function_output, ensure_ascii=False),
                }
            ],
            previous_response_id=first_response.id,
        )
    except Exception as error:
        return {"status": "error", "message": f"Second-stage error: {extract_error_details(error)}"}

    return {
        "status": "ok",
        "function_name": function_name,
        "function_arguments": arguments,
        "function_result": function_output,
        "final_recommendation": getattr(second_response, "output_text", "No textual output"),
    }

In [71]:
flow_result = run_grounded_financial_flow()

print("Flow status:", flow_result.get("status"))
if flow_result.get("status") in {"ok", "demo"}:
    if flow_result.get("status") == "demo":
        print("\nUsing cached market snapshot to avoid throttling.")
        print("\nMarket context:\n")
        print(flow_result.get("market_context"))
    print("\nArguments used in calcular_viabilidad_financiera:")
    print(json.dumps(flow_result.get("function_arguments"), ensure_ascii=False, indent=2))
    print("\nResult from calcular_viabilidad_financiera:")
    print(json.dumps(flow_result.get("function_result"), ensure_ascii=False, indent=2))
    print("\nFinal strategic recommendation (model):\n")
    print(flow_result.get("final_recommendation", "No recommendation"))
else:
    print(flow_result.get("message", "The flow could not be completed"))

Flow status: demo

Using cached market snapshot to avoid throttling.

Market context:

### Cached market snapshot (2026)

- premium fish: trend=up, pressure=0.18. Higher energy and logistics costs are keeping imported premium seafood expensive.
- olive oil: trend=up, pressure=0.12. Supply volatility continues to push retail and wholesale prices upward.
- dairy: trend=flat, pressure=0.04. Costs are stable but remain sensitive to feed and transport constraints.
- energy: trend=up, pressure=0.15. Energy remains a structural cost driver across the restaurant supply chain.

Arguments used in calcular_viabilidad_financiera:
{
  "coste": 7.4,
  "precio_venta": 15.9,
  "demanda_estimada": 1200
}

Result from calcular_viabilidad_financiera:
{
  "escenario": "margen_positivo",
  "coste_unitario": 7.4,
  "precio_venta_unitario": 15.9,
  "demanda_estimada": 1200,
  "margen_unitario": 8.5,
  "ingreso_estimado": 19080.0,
  "coste_total": 8880.0,
  "beneficio_estimado": 10200.0,
  "rentabilidad_sobre

## Conclusiones y problemas encontrados

El notebook cumple la comparativa de razonamiento y muestra un flujo de function calling con evidencia de mercado, función custom y recomendación final.

Problemas encontrados durante la elaboración: límites de cuota del servicio, ajustes del formato de `responses.create` y necesidad de un modo demo para evitar esperas excesivas en la entrega.

La versión actual prioriza reproducibilidad y experiencia de evaluación: el modo demo responde rápido, y la ruta en vivo queda disponible para validación adicional cuando la cuota lo permita.